## 1. 라이브러리 임포트 및 파일 경로 설정

In [1]:
"""
===================================================================================
1. 라이브러리 임포트 및 파일 경로 설정
===================================================================================
목적: 편의시설온도 계산에 필요한 라이브러리와 데이터 파일 경로를 설정합니다.

주요 라이브러리:
- pandas: 데이터프레임 처리
- numpy: 수치 연산
- json: JSON 파일 읽기
- haversine 관련 함수: 위도/경도로 거리 계산
===================================================================================
"""

import pandas as pd
import numpy as np
import json
from math import radians, sin, cos, asin, sqrt
from pathlib import Path

# 파일 경로 설정
# BASE_DIR: 프로젝트 루트 디렉토리
BASE_DIR = Path(r"c:\Users\Playdata\OneDrive\바탕 화면\ondo")

AMENITY_DIR = BASE_DIR / "ondo_conveient" / "data" / "상가(상권)정보"
LAND_DIR = BASE_DIR / "land_data" / "land"
CONVENIENCE_CSV = AMENITY_DIR / "소상공인시장진흥공단_상가(상권)정보_서울_cleaned.csv"
HOSPITAL_XLSX = AMENITY_DIR / "1.병원정보서비스(2025.9).xlsx"
PHARMACY_XLSX = AMENITY_DIR / "2.약국정보서비스(2025.9).xlsx"

LISTING_JSON = LAND_DIR / "00_통합_원투룸_2.json"

print(f"\n데이터 폴더: {AMENITY_DIR}")
print(f"매물 파일: {LISTING_JSON.name}")


데이터 폴더: c:\Users\Playdata\OneDrive\바탕 화면\ondo\ondo_conveient\data\상가(상권)정보
매물 파일: 00_통합_원투룸_2.json


## 2. POI/병원/약국 데이터 로드

In [2]:
"""
===================================================================================
2. POI(편의시설)/병원/약국 데이터 로드
===================================================================================
목적:
1. 서울시 전체 편의시설 데이터 로드 및 필터링
2. 병원, 약국 데이터 로드
3. 유효한 좌표 데이터만 선별

데이터 규모:
- 상가: 53만개 → 편의시설 8.2만개 필터링
- 병원: 1.9만개 (서울)
- 약국: 5,790개 (서울)
===================================================================================
"""

print("\n" + "=" * 60)
print("1. 편의시설 POI 데이터 로드")
print("=" * 60)

# === 1-1. 상가(편의시설) 데이터 ===
# 소상공인시장진흥공단 상가 데이터 로드
convenience_df = pd.read_csv(CONVENIENCE_CSV, encoding="utf-8")
print(f"\n전체 상가 데이터: {len(convenience_df)}개")
print(f"컬럼: {convenience_df.columns.tolist()}")

# 편의시설 업종만 필터링
# 일상 생활에 자주 이용하는 시설 8가지 선정
CONVENIENCE_TYPES = [
    "편의점",          # CU, GS25, 세븐일레븐 등
    "슈퍼마켓",        # 동네 슈퍼
    "카페",            # 스타벅스, 이디야 등
    "백반/한정식",     # 일반 식당
    "김밥/만두/분식",  # 간단한 식사
    "빵/도넛",         # 베이커리
    "세탁소",          # 세탁 서비스
    "셀프 빨래방"      # 코인 빨래방
]

# 업종 필터링
convenience_poi = convenience_df[
    convenience_df["상권업종소분류명"].isin(CONVENIENCE_TYPES)
].copy()

print(f"\n편의시설 필터링 완료: {len(convenience_poi)}개")
print(f"업종별 분포:\n{convenience_poi['상권업종소분류명'].value_counts()}")

# === 1-2. 병원 데이터 ===
# 건강보험심사평가원 병원 정보 로드
hospital_df = pd.read_excel(HOSPITAL_XLSX)
print(f"\n\n병원 데이터 로드: {len(hospital_df)}개")
print(f"컬럼: {hospital_df.columns.tolist()}")

# 서울만 필터링
hospital_poi = hospital_df[hospital_df["시도코드명"] == "서울"].copy()

# 컬럼명 통일 (X=경도, Y=위도)
hospital_poi = hospital_poi.rename(columns={"좌표(X)": "경도", "좌표(Y)": "위도"})
print(f"서울 병원: {len(hospital_poi)}개")

# === 1-3. 약국 데이터 ===
# 건강보험심사평가원 약국 정보 로드
pharmacy_df = pd.read_excel(PHARMACY_XLSX)
print(f"\n약국 데이터 로드: {len(pharmacy_df)}개")

# 서울만 필터링
pharmacy_poi = pharmacy_df[pharmacy_df["시도코드명"] == "서울"].copy()

# 컬럼명 통일
pharmacy_poi = pharmacy_poi.rename(columns={"좌표(X)": "경도", "좌표(Y)": "위도"})
print(f"서울 약국: {len(pharmacy_poi)}개")

# === 유효한 좌표만 사용 ===
# NaN 값이 있는 행 제거 (좌표가 없으면 거리 계산 불가)
convenience_poi = convenience_poi[convenience_poi[["경도", "위도"]].notna().all(axis=1)]
hospital_poi = hospital_poi[hospital_poi[["경도", "위도"]].notna().all(axis=1)]
pharmacy_poi = pharmacy_poi[pharmacy_poi[["경도", "위도"]].notna().all(axis=1)]

print(f"\n유효 좌표 필터링 후:")
print(f"  편의시설: {len(convenience_poi)}개")  # 82,559개
print(f"  병원: {len(hospital_poi)}개")         # 19,594개
print(f"  약국: {len(pharmacy_poi)}개")         # 5,790개


1. 편의시설 POI 데이터 로드

전체 상가 데이터: 536262개
컬럼: ['상가업소번호', '상호명', '상권업종소분류명', '경도', '위도']

편의시설 필터링 완료: 82559개
업종별 분포:
상권업종소분류명
백반/한정식      24771
카페          21909
편의점          9597
슈퍼마켓         8081
김밥/만두/분식     7838
빵/도넛         5752
세탁소          3789
셀프 빨래방        822
Name: count, dtype: int64


병원 데이터 로드: 79322개
컬럼: ['암호화요양기호', '요양기관명', '종별코드', '종별코드명', '시도코드', '시도코드명', '시군구코드', '시군구코드명', '읍면동', '우편번호', '주소', '전화번호', '병원홈페이지', '개설일자', '총의사수', '의과일반의 인원수', '의과인턴 인원수', '의과레지던트 인원수', '의과전문의 인원수', '치과일반의 인원수', '치과인턴 인원수', '치과레지던트 인원수', '치과전문의 인원수', '한방일반의 인원수', '한방인턴 인원수', '한방레지던트 인원수', '한방전문의 인원수', '조산사 인원수', '좌표(X)', '좌표(Y)']
서울 병원: 19594개

약국 데이터 로드: 25514개
서울 약국: 5791개

유효 좌표 필터링 후:
  편의시설: 82559개
  병원: 19594개
  약국: 5790개


## 3. 매물 데이터 로드

In [4]:
"""
===================================================================================
3. 매물 데이터 로드
===================================================================================
목적:
1. JSON 형식의 부동산 매물 데이터 로드
2. 매물번호, 주소, 생활시설 정보 추출
3. 생활시설 정보는 필요도(Need) 계산에 사용됨

생활시설 정보:
- 냉장고, 세탁기, 싱크대, 가스레인지, 인덕션 등
- 이 정보로 매물의 옵션 수준을 파악
- 옵션이 부족할수록 외부 편의시설 의존도 증가
===================================================================================
"""

print("\n" + "=" * 60)
print("2. 매물 데이터 로드 및 지오코딩")
print("=" * 60)

# JSON 파일 로드
with open(LISTING_JSON, encoding="utf-8") as f:
    listings_data = json.load(f)

print(f"총 매물 수: {len(listings_data)}개")

# 매물 데이터프레임 생성
all_listings = []

for item in listings_data:
    # 매물번호 추출
    listing_id = item.get("매물번호", "")
    
    # 주소 정보 추출
    address_info = item.get("주소_정보", {})
    address = address_info.get("전체주소", "")
    
    # 생활시설 정보 추출 (필요도 계산에 사용)
    # 예: "냉장고, 세탁기, 싱크대, 가스레인지, ..."
    property_info = item.get("매물_정보", {})
    facilities = property_info.get("생활시설", "")
    
    # 매물번호와 주소가 모두 있는 경우만 포함
    if listing_id and address:
        all_listings.append({
            "listing_id": listing_id,
            "address": address,
            "facilities": facilities if facilities else "",  # 생활시설 정보
            "lat": None,  # 지오코딩 후 채워질 예정
            "lng": None,  # 지오코딩 후 채워질 예정
        })

# 리스트를 DataFrame으로 변환
listings = pd.DataFrame(all_listings)

print(f"\n매물 데이터프레임 생성: {len(listings)}개")
print(listings.head())


2. 매물 데이터 로드 및 지오코딩
총 매물 수: 6개

매물 데이터프레임 생성: 6개
  listing_id                        address  \
0   18446002      서울특별시 서초구 양재동 116-3 소호M타워   
1   18350009     서울특별시 서초구 양재동 208-9 화평빌라타운   
2   18448077  서울특별시 서초구 양재동 257-8 다비치 스위트 홈   
3   18320532       서울특별시 서초구 양재동 342-8 리더스빌   
4   18306808            서울특별시 서초구 양재동 366-9   

                                         facilities   lat   lng  
0  냉장고, 세탁기, 싱크대, 옷장, 붙박이장, 신발장, 인덕션레인지, 가스오븐, 샤워부스  None  None  
1                                   신발장, 싱크대, 가스레인지  None  None  
2                냉장고, 세탁기, 싱크대, 침대, 신발장, 인덕션레인지, TV  None  None  
3                              냉장고, 세탁기, 싱크대, 가스레인지  None  None  
4    냉장고, 세탁기, 싱크대, 신발장, 전자레인지, 가스레인지, 샤워부스, 비데, 식탁  None  None  


## 4. 지오코딩 

In [ ]:
"""
===================================================================================
4. 지오코딩 (주소 → 위도/경도 변환)
===================================================================================
목적:
1. 카카오 API를 사용해서 텍스트 주소를 위도/경도로 변환
2. 기존 지오코딩 파일이 있으면 재사용 (API 호출 절약)
3. API 호출 제한을 고려한 속도 조절 (0.1초 대기)

주의사항:
- 카카오 API 무료 플랜: 일일 호출 제한 존재
- 다른 노트북(교통/공원온도)에서 생성한 파일 재사용 가능
===================================================================================
"""

print("\n" + "=" * 60)
print("2-1. 지오코딩")
print("=" * 60)

import requests
import time

# 카카오 지오코딩 함수 정의
def geocode_address(address, kakao_api_key=None):
    """
    카카오 REST API를 사용해서 주소를 위도/경도로 변환
    
    Args:
        address: 검색할 주소 (예: "서울특별시 서초구 양재동 116-3")
        kakao_api_key: 카카오 개발자 앱의 REST API 키
    
    Returns:
        (위도, 경도) 튜플, 또는 실패 시 (None, None)
    """
    # API 키가 없으면 지오코딩 불가
    if not kakao_api_key:
        return None, None
    
    # 카카오 로컬 API 엔드포인트
    url = "https://dapi.kakao.com/v2/local/search/address.json"
    
    # API 인증 헤더 (REST API 키 사용)
    headers = {"Authorization": f"KakaoAK {kakao_api_key}"}
    
    # 검색 파라미터
    params = {"query": address}
    
    try:
        # GET 요청 전송 (timeout 5초)
        response = requests.get(url, headers=headers, params=params, timeout=5)
        
        # 성공 응답 확인 (HTTP 200)
        if response.status_code == 200:
            result = response.json()
            
            # 검색 결과가 있으면 첫 번째 결과 사용
            if result['documents']:
                doc = result['documents'][0]
                # y = 위도(latitude), x = 경도(longitude)
                return float(doc['y']), float(doc['x'])
    except:
        pass
    
    return None, None

# 카카오 API 키 설정
KAKAO_API_KEY = ""

# 기존 지오코딩 파일 확인 (API 호출 절약)
# 다른 노트북에서 생성한 파일도 재사용 가능
saved_files = [
    BASE_DIR / "amenity_listings_with_coords.csv",  # 이 노트북에서 생성
    BASE_DIR / "listings_with_coords.csv",          # 교통온도에서 생성
    BASE_DIR / "park_listings_with_coords.csv",     # 공원온도에서 생성
]

coords_loaded = False

# 기존 파일 찾기
for saved_file in saved_files:
    if saved_file.exists():
        print(f"\n기존 지오코딩 파일 발견: {saved_file.name}")
        
        try:
            # 기존 파일 로드
            coords_df = pd.read_csv(saved_file, encoding="utf-8-sig")
            
            # 좌표 병합 (listing_id 기준)
            if "listing_id" in coords_df.columns and "lat" in coords_df.columns:
                listings = listings.merge(
                    coords_df[["listing_id", "lat", "lng"]], 
                    on="listing_id", 
                    how="left",
                    suffixes=("_old", "")  # 충돌 시 새로운 좌표 사용
                )
                
                # 기존 좌표 컬럼 제거
                if "lat_old" in listings.columns:
                    listings = listings.drop(columns=["lat_old", "lng_old"], errors="ignore")
                
                coords_loaded = True
                print(f"좌표 데이터 로드 완료!")
                break
                
        except Exception as e:
            print(f"파일 로드 실패: {e}")

# 카카오 API로 지오코딩 수행 (기존 파일이 없는 경우)
if not coords_loaded and KAKAO_API_KEY:
    print("\n카카오 API를 사용해서 지오코딩을 수행합니다...")
    print(f"총 {len(listings)}개 주소 처리 예정")
    
    # 각 매물의 주소를 지오코딩
    for idx, row in listings.iterrows():
        # 진행 상황 출력 (100개마다)
        if idx % 100 == 0:
            print(f"  진행: {idx}/{len(listings)}")
        
        # 카카오 API 호출하여 위도/경도 받기
        lat, lng = geocode_address(row['address'], KAKAO_API_KEY)
        
        # DataFrame에 저장
        listings.at[idx, 'lat'] = lat
        listings.at[idx, 'lng'] = lng
        
        # API 호출 제한 방지: 0.1초 대기
        time.sleep(0.1)
    
    # 지오코딩 성공률 확인
    success_count = listings[['lat', 'lng']].notna().all(axis=1).sum()
    print(f"\n지오코딩 성공: {success_count}/{len(listings)} ({success_count/len(listings)*100:.1f}%)")
    
    # 지오코딩된 데이터 저장
    listings.to_csv(BASE_DIR / "amenity_listings_with_coords.csv", index=False, encoding="utf-8-sig")
    print("지오코딩 결과를 'amenity_listings_with_coords.csv'에 저장했습니다.")
    
elif not coords_loaded and not KAKAO_API_KEY:
    # API 키도 없고 기존 파일도 없는 경우
    print("\n⚠️ 경고: 카카오 API 키가 설정되지 않았고, 기존 지오코딩 파일도 없습니다.")
    print("\n다음 중 하나를 선택하세요:")
    print("1. 위 셀에서 KAKAO_API_KEY 변수에 카카오 API 키를 입력")
    print("2. 또는 다른 노트북(교통온도/공원온도)에서 먼저 지오코딩 수행")
    print("\n현재는 샘플 데이터로 진행합니다 (좌표가 없는 매물은 제외됩니다).")

# 유효한 좌표가 있는 매물만 필터링
# lat과 lng가 모두 NaN이 아닌 행만 선택
listings_valid = listings[listings[['lat', 'lng']].notna().all(axis=1)].copy()

print(f"\n유효한 좌표가 있는 매물: {len(listings_valid)}개 / {len(listings)}개")

# 결과 확인
if len(listings_valid) > 0:
    print("\n샘플 데이터:")
    print(listings_valid[["listing_id", "address", "lat", "lng"]].head())
else:
    print("\n⚠️ 경고: 유효한 좌표가 있는 매물이 없습니다!")
    print("지오코딩을 먼저 수행해주세요.")


2-1. 지오코딩

기존 지오코딩 파일 발견: amenity_listings_with_coords.csv
파일 로드 실패: You are trying to merge on object and int64 columns for key 'listing_id'. If you wish to proceed you should use pd.concat

기존 지오코딩 파일 발견: listings_with_coords.csv
파일 로드 실패: You are trying to merge on object and int64 columns for key 'listing_id'. If you wish to proceed you should use pd.concat

기존 지오코딩 파일 발견: park_listings_with_coords.csv
파일 로드 실패: You are trying to merge on object and int64 columns for key 'listing_id'. If you wish to proceed you should use pd.concat

카카오 API를 사용해서 지오코딩을 수행합니다...
총 6개 주소 처리 예정
  진행: 0/6

지오코딩 성공: 6/6 (100.0%)
지오코딩 결과를 'amenity_listings_with_coords.csv'에 저장했습니다.

유효한 좌표가 있는 매물: 6개 / 6개

샘플 데이터:
  listing_id                        address        lat         lng
0   18446002      서울특별시 서초구 양재동 116-3 소호M타워  37.474678  127.035655
1   18350009     서울특별시 서초구 양재동 208-9 화평빌라타운  37.465608  127.034504
2   18448077  서울특별시 서초구 양재동 257-8 다비치 스위트 홈  37.474999  127.041886
3   18320532       서울특별시 서

## 5. 편의시설 읜존도(필요도) 계산

In [6]:
"""
===================================================================================
5. 필요도(Need) 계산
===================================================================================
목적:
1. 매물의 생활시설(옵션)을 분석하여 외부 편의시설 의존도를 계산
2. 옵션이 부족할수록 Need 값이 높아짐 (외부 시설 더 필요)

핵심 개념:
- 같은 위치라도 매물 옵션에 따라 편의시설 중요도가 다름
- 예: 세탁기 없으면 세탁소 필수, 주방 없으면 식당 필수

필요도 계산 로직:
===================================================================================
"""

print("\n" + "=" * 60)
print("3. 필요도(Need) 계산")
print("=" * 60)

def calculate_need_convenience(facilities_str):
    """
    편의시설 필요도 계산
    
    Args:
        facilities_str: 생활시설 문자열 (예: "냉장고, 세탁기, 싱크대, 가스레인지")
    
    Returns:
        Need 값 (0.7~1.0)
    
    계산 로직:
    - 기본값: 0.7 (일반적인 필요도)
    - 세탁기 없음: +0.1 (세탁소 의존도 증가)
    - 부실 주방: +0.1 (식당 의존도 증가)
    - 최대값: 1.0 (외부 시설에 매우 의존적)
    
    부실 주방 판정:
    - 냉장고 없음 OR (싱크대 없음 AND 가스/인덕션 없음)
    """
    need = 0.7  # 기본 필요도
    
    # 소문자로 변환 (대소문자 구분 없이 검색)
    facilities_lower = str(facilities_str).lower()
    
    # === 1. 세탁기 확인 ===
    if "세탁기" not in facilities_lower:
        need += 0.1  # 세탁기 없으면 세탁소/빨래방 필수
    
    # === 2. 주방 시설 확인 ===
    # 냉장고, 싱크대, 가스/인덕션 중 2개 이상 없으면 부실 주방
    has_fridge = "냉장고" in facilities_lower
    has_sink = "싱크대" in facilities_lower
    has_stove = "가스" in facilities_lower or "인덕션" in facilities_lower
    
    # 부실 주방 판정
    if not has_fridge or (not has_sink and not has_stove):
        need += 0.1  # 주방 부실하면 식당/편의점 의존도 증가
    
    # 최대값 1.0으로 제한
    return min(1.0, need)

# 매물별 필요도 계산
if len(listings_valid) > 0:
    # 편의시설 필요도 계산
    listings_valid["Need_conv"] = listings_valid["facilities"].apply(calculate_need_convenience)
    
    # 병원/약국 필요도는 고정값 (매물 옵션과 무관)
    listings_valid["Need_med"] = 0.7    # 병원 필요도 고정
    listings_valid["Need_pharm"] = 0.7  # 약국 필요도 고정
    
    print("필요도 계산 완료!")
    
    # 통계 출력
    print("\n편의시설 필요도 통계:")
    print(listings_valid["Need_conv"].describe())
    
    # 필요도별 분포 확인
    print("\n필요도별 매물 수:")
    print(listings_valid["Need_conv"].value_counts().sort_index())
    
    # 샘플 출력 (생활시설 → 필요도 매핑 확인)
    print("\n샘플 (생활시설 → 필요도):")
    sample = listings_valid[["listing_id", "facilities", "Need_conv"]].head(10)
    
    for _, row in sample.iterrows():
        print(f"  매물 {row['listing_id']}: Need={row['Need_conv']:.1f}")
        # 생활시설 문자열이 길면 80자까지만 출력
        facilities_display = row['facilities'][:80] + "..." if len(row['facilities']) > 80 else row['facilities']
        print(f"    시설: {facilities_display}")
        
        # 점수 해석
        if row['Need_conv'] == 0.7:
            print(f"    → 풀옵션 (외부 시설 의존도 낮음)")
        elif row['Need_conv'] == 0.8:
            print(f"    → 세탁기 또는 주방 중 1개 부족")
        elif row['Need_conv'] == 0.9:
            print(f"    → 세탁기 + 주방 모두 부족 (외부 시설 의존도 높음)")
        else:
            print(f"    → 옵션 부족 심각 (편의시설 매우 필요)")
            
else:
    print("유효한 매물이 없습니다.")


3. 필요도(Need) 계산
필요도 계산 완료!

편의시설 필요도 통계:
count    6.000000
mean     0.733333
std      0.081650
min      0.700000
25%      0.700000
50%      0.700000
75%      0.700000
max      0.900000
Name: Need_conv, dtype: float64

필요도별 매물 수:
Need_conv
0.7    5
0.9    1
Name: count, dtype: int64

샘플 (생활시설 → 필요도):
  매물 18446002: Need=0.7
    시설: 냉장고, 세탁기, 싱크대, 옷장, 붙박이장, 신발장, 인덕션레인지, 가스오븐, 샤워부스
    → 풀옵션 (외부 시설 의존도 낮음)
  매물 18350009: Need=0.9
    시설: 신발장, 싱크대, 가스레인지
    → 옵션 부족 심각 (편의시설 매우 필요)
  매물 18448077: Need=0.7
    시설: 냉장고, 세탁기, 싱크대, 침대, 신발장, 인덕션레인지, TV
    → 풀옵션 (외부 시설 의존도 낮음)
  매물 18320532: Need=0.7
    시설: 냉장고, 세탁기, 싱크대, 가스레인지
    → 풀옵션 (외부 시설 의존도 낮음)
  매물 18306808: Need=0.7
    시설: 냉장고, 세탁기, 싱크대, 신발장, 전자레인지, 가스레인지, 샤워부스, 비데, 식탁
    → 풀옵션 (외부 시설 의존도 낮음)
  매물 18443369: Need=0.7
    시설: 냉장고, 세탁기, 싱크대, 책상, 옷장, 침대, 신발장, 인덕션레인지
    → 풀옵션 (외부 시설 의존도 낮음)


## 6. 거리계산 및 접근성 함수

In [7]:
"""
===================================================================================
6. 거리 계산 및 접근성(Access) 함수
===================================================================================
목적:
1. 위도/경도로 두 지점 간 실제 거리 계산 (Haversine 공식)
2. 접근성(Access) 점수 계산 함수 정의

접근성 = 0.6 × 거리 점수 + 0.4 × 밀도 점수

- 거리 점수: 가장 가까운 POI까지 거리
- 밀도 점수: 반경 내 POI 개수
===================================================================================
"""

print("\n" + "=" * 60)
print("4. 거리 계산 및 접근성 함수")
print("=" * 60)

def haversine(lat1, lon1, lat2, lon2):
    """
    Haversine 공식을 사용한 두 지점 간 거리 계산
    
    Args:
        lat1, lon1: 첫 번째 지점의 위도, 경도 (도 단위)
        lat2, lon2: 두 번째 지점의 위도, 경도 (도 단위)
    
    Returns:
        거리 (미터 단위)
    
    공식 설명:
    1. 위도/경도를 라디안으로 변환
    2. Haversine 공식으로 중심각 계산
    3. 중심각 × 지구 반지름 = 거리
    """
    R = 6371000  # 지구 반지름 (미터)
    
    # 도(degree)를 라디안(radian)으로 변환
    lat1, lon1, lat2, lon2 = map(radians, [lat1, lon1, lat2, lon2])
    
    # 위도/경도 차이
    dlat = lat2 - lat1
    dlon = lon2 - lon1
    
    # Haversine 공식
    a = sin(dlat/2)**2 + cos(lat1)*cos(lat2)*sin(dlon/2)**2
    c = 2 * asin(sqrt(a))  # 중심각
    
    return R * c  # 거리 = 반지름 × 중심각

def distance_score(d, d_max):
    """
    거리 점수 계산
    
    Args:
        d: 실제 거리 (미터)
        d_max: 기준 거리 (미터)
    
    Returns:
        거리 점수 (0~1)
    
    공식:
        distance_score = max(0, 1 - d/d_max)
    
    예시 (d_max = 800m):
    - d = 0m   → 1.0 (만점)
    - d = 400m → 0.5 (중간)
    - d = 800m → 0.0 (경계)
    - d > 800m → 0.0 (너무 멀음)
    """
    return max(0.0, 1.0 - d / d_max)

def density_score(n, n_sat):
    """
    밀도 점수 계산
    
    Args:
        n: 실제 POI 개수
        n_sat: 포화 개수 (이 개수 이상이면 만점)
    
    Returns:
        밀도 점수 (0~1)
    
    공식:
        density_score = min(n/n_sat, 1.0)
    
    예시 (n_sat = 5):
    - n = 0개 → 0.0 (없음)
    - n = 2개 → 0.4 (부족)
    - n = 5개 → 1.0 (충분, 만점)
    - n = 10개 → 1.0 (과잉, 만점 유지)
    """
    return min(n / n_sat, 1.0)

def calculate_access_score(p_lat, p_lng, poi_coords, d_max, radius, n_sat):
    """
    접근성 점수 계산
    
    Args:
        p_lat, p_lng: 매물 좌표
        poi_coords: POI 좌표 배열 (numpy array)
        d_max: 거리 점수 기준 (미터)
        radius: 밀도 계산 반경 (미터)
        n_sat: 밀도 포화 개수
    
    Returns:
        접근성 점수 (0~1)
    
    공식:
        Access = 0.6 × distance_score + 0.4 × density_score
    
    알고리즘:
    1. 매물에서 모든 POI까지 거리 계산
    2. 거리 점수 = max(0, 1 - d_min/d_max)
    3. 밀도 점수 = min(n_in_radius/n_sat, 1.0)
    4. 접근성 = 0.6×거리 + 0.4×밀도
    
    가중치 근거:
    - 거리 60%: 가장 가까운 POI가 중요
    - 밀도 40%: 선택의 다양성 반영
    """
    # POI 데이터가 없으면 0점
    if len(poi_coords) == 0:
        return 0.0
    
    # 매물에서 모든 POI까지의 거리 계산
    dists = np.array([
        haversine(p_lat, p_lng, poi_lat, poi_lng)
        for (poi_lat, poi_lng) in poi_coords
    ])
    
    # 1. 거리 점수: 가장 가까운 POI까지 거리
    d_min = dists.min()
    d_score = distance_score(d_min, d_max)
    
    # 2. 밀도 점수: 반경 내 POI 개수
    n_in_radius = (dists <= radius).sum()
    dens_score = density_score(n_in_radius, n_sat)
    
    # 3. 접근성 = 0.6 × 거리 + 0.4 × 밀도
    access = 0.6 * d_score + 0.4 * dens_score
    
    return float(access)

# POI 좌표를 numpy 배열로 변환 (속도 최적화)
# 매물 1개당 수만개 POI와의 거리를 계산하므로 속도가 중요
conv_coords = convenience_poi[["위도", "경도"]].to_numpy()
hosp_coords = hospital_poi[["위도", "경도"]].to_numpy()
pharm_coords = pharmacy_poi[["위도", "경도"]].to_numpy()

print("POI 좌표 배열 변환 완료")
print(f"  편의시설: {len(conv_coords)}개")
print(f"  병원: {len(hosp_coords)}개")
print(f"  약국: {len(pharm_coords)}개")

# 접근성 계산 파라미터 설정
PARAMS = {
    "convenience": {
        "d_max": 800,    # 거리 기준: 800m (도보 10분)
        "radius": 500,   # 밀도 반경: 500m
        "n_sat": 5       # 포화 개수: 5개
    },
    "hospital": {
        "d_max": 1200,   # 거리 기준: 1200m (도보 15분)
        "radius": 800,   # 밀도 반경: 800m
        "n_sat": 2       # 포화 개수: 2개 (병원은 적어도 됨)
    },
    "pharmacy": {
        "d_max": 1200,   # 거리 기준: 1200m
        "radius": 800,   # 밀도 반경: 800m
        "n_sat": 2       # 포화 개수: 2개
    },
}

print("\n접근성 계산 파라미터:")
for key, val in PARAMS.items():
    print(f"  {key}: {val}")


4. 거리 계산 및 접근성 함수
POI 좌표 배열 변환 완료
  편의시설: 82559개
  병원: 19594개
  약국: 5790개

접근성 계산 파라미터:
  convenience: {'d_max': 800, 'radius': 500, 'n_sat': 5}
  hospital: {'d_max': 1200, 'radius': 800, 'n_sat': 2}
  pharmacy: {'d_max': 1200, 'radius': 800, 'n_sat': 2}


## 7. 매물별로 계산

In [8]:
"""
===================================================================================
7. 매물별 접근성(Access) 계산
===================================================================================
목적:
1. 각 매물에 대해 편의시설/병원/약국 접근성 계산
2. 수만개 POI와의 거리를 계산하므로 시간 소요

계산 결과:
- Access_conv: 편의시설 접근성 (0~1)
- Access_med: 병원 접근성 (0~1)
- Access_pharm: 약국 접근성 (0~1)
===================================================================================
"""

print("\n" + "=" * 60)
print("5. 매물별 접근성(Access) 계산")
print("=" * 60)

if len(listings_valid) > 0:
    print("매물별 접근성 점수 계산 중...")
    print("(수만개 POI와의 거리 계산으로 시간 소요)\n")
    
    # 결과 저장 리스트
    access_conv_list = []
    access_med_list = []
    access_pharm_list = []
    
    for idx, row in listings_valid.iterrows():
        # 진행 상황 출력 (500개마다)
        if idx % 500 == 0:
            print(f"  진행: {idx}/{len(listings_valid)}")
        
        # === 1. 편의시설 접근성 ===
        access_conv = calculate_access_score(
            row["lat"], row["lng"], 
            conv_coords,
            PARAMS["convenience"]["d_max"],    # 800m
            PARAMS["convenience"]["radius"],   # 500m
            PARAMS["convenience"]["n_sat"]     # 5개
        )
        access_conv_list.append(access_conv)
        
        # === 2. 병원 접근성 ===
        access_med = calculate_access_score(
            row["lat"], row["lng"], 
            hosp_coords,
            PARAMS["hospital"]["d_max"],       # 1200m
            PARAMS["hospital"]["radius"],      # 800m
            PARAMS["hospital"]["n_sat"]        # 2개
        )
        access_med_list.append(access_med)
        
        # === 3. 약국 접근성 ===
        access_pharm = calculate_access_score(
            row["lat"], row["lng"], 
            pharm_coords,
            PARAMS["pharmacy"]["d_max"],       # 1200m
            PARAMS["pharmacy"]["radius"],      # 800m
            PARAMS["pharmacy"]["n_sat"]        # 2개
        )
        access_pharm_list.append(access_pharm)
    
    # DataFrame에 저장
    listings_valid["Access_conv"] = access_conv_list
    listings_valid["Access_med"] = access_med_list
    listings_valid["Access_pharm"] = access_pharm_list
    
    print("\n접근성 계산 완료!")
    
    # 통계 출력
    print("\n접근성 점수 통계:")
    print("편의시설:")
    print(listings_valid["Access_conv"].describe())
    print("\n병원:")
    print(listings_valid["Access_med"].describe())
    print("\n약국:")
    print(listings_valid["Access_pharm"].describe())
    
    # 해석
    print("\n결과 해석:")
    print(f"  편의시설 평균: {listings_valid['Access_conv'].mean():.3f}")
    print(f"    → 서울은 편의시설이 매우 잘 갖춰져 있음")
    print(f"  병원 평균: {listings_valid['Access_med'].mean():.3f}")
    print(f"    → 병원 접근성도 양호")
    print(f"  약국 평균: {listings_valid['Access_pharm'].mean():.3f}")
    print(f"    → 약국도 적절히 분포")
    
else:
    print("유효한 매물이 없습니다.")


5. 매물별 접근성(Access) 계산
매물별 접근성 점수 계산 중...
(수만개 POI와의 거리 계산으로 시간 소요)

  진행: 0/6

접근성 계산 완료!

접근성 점수 통계:
편의시설:
count    6.000000
mean     0.972653
std      0.006767
min      0.961107
25%      0.970177
50%      0.974269
75%      0.977409
max      0.979002
Name: Access_conv, dtype: float64

병원:
count    6.000000
mean     0.893728
std      0.039391
min      0.833705
25%      0.868364
50%      0.915885
75%      0.919835
max      0.923218
Name: Access_med, dtype: float64

약국:
count    6.000000
mean     0.874267
std      0.086464
min      0.738258
25%      0.840053
50%      0.875994
75%      0.927738
max      0.982300
Name: Access_pharm, dtype: float64

결과 해석:
  편의시설 평균: 0.973
    → 서울은 편의시설이 매우 잘 갖춰져 있음
  병원 평균: 0.894
    → 병원 접근성도 양호
  약국 평균: 0.874
    → 약국도 적절히 분포


## 8. 최종 편의시설 온도 계산

In [9]:
"""
===================================================================================
8. 최종 편의시설온도 계산
===================================================================================
목적:
1. Need × Access로 유형별 점수 계산
2. 가중치 적용하여 편의시설 총점 산출
3. 30~43°C 온도로 변환

핵심 로직:
- 유형별 점수: S_type = Need × Access
  - S_conv = Need_conv × Access_conv (편의시설)
  - S_med = Need_med × Access_med (병원)
  - S_pharm = Need_pharm × Access_pharm (약국)

- 총점: S_amenity = 0.5×S_conv + 0.25×S_med + 0.25×S_pharm
  - 편의시설 50%: 일상 생활에서 가장 자주 이용
  - 병원 25%: 중요하지만 자주 이용하지 않음
  - 약국 25%: 필요할 때 중요

- 온도: AmenityTemp = 30 + 13 × S_amenity
  - S_amenity = 0 → 30°C (최저)
  - S_amenity = 0.5 → 36.5°C (평균 체온)
  - S_amenity = 1 → 43°C (최고)
===================================================================================
"""

print("\n" + "=" * 60)
print("6. 최종 편의시설온도 계산")
print("=" * 60)

if len(listings_valid) > 0:
    # ========================================
    # 6-1. 유형별 점수 계산: S_type = Need × Access
    # ========================================
    
    # 편의시설 점수 = 필요도 × 접근성
    # 예: Need=0.9(옵션부족), Access=0.8(접근양호) → S=0.72
    listings_valid["S_conv"] = listings_valid["Need_conv"] * listings_valid["Access_conv"]
    
    # 병원 점수 = 필요도 × 접근성
    # Need는 고정값 0.7 사용
    listings_valid["S_med"] = listings_valid["Need_med"] * listings_valid["Access_med"]
    
    # 약국 점수 = 필요도 × 접근성
    # Need는 고정값 0.7 사용
    listings_valid["S_pharm"] = listings_valid["Need_pharm"] * listings_valid["Access_pharm"]
    
    print("유형별 점수 계산 완료 (S = Need × Access)")
    print(f"  S_conv (편의시설): 평균 {listings_valid['S_conv'].mean():.3f}")
    print(f"  S_med (병원): 평균 {listings_valid['S_med'].mean():.3f}")
    print(f"  S_pharm (약국): 평균 {listings_valid['S_pharm'].mean():.3f}")
    
    # ========================================
    # 6-2. 편의시설 총점: 가중 평균
    # ========================================
    
    # 가중치 설정
    W_CONV = 0.5    # 편의시설 50%: 일상 생활에서 가장 자주 이용
    W_MED = 0.25    # 병원 25%: 중요하지만 자주 이용하지 않음
    W_PHARM = 0.25  # 약국 25%: 필요할 때 중요
    
    # 총점 = 0.5×편의 + 0.25×병원 + 0.25×약국
    # 예: S_conv=0.7, S_med=0.6, S_pharm=0.6
    #     → S_amenity = 0.5×0.7 + 0.25×0.6 + 0.25×0.6 = 0.65
    listings_valid["S_amenity"] = (
        W_CONV * listings_valid["S_conv"] +
        W_MED * listings_valid["S_med"] +
        W_PHARM * listings_valid["S_pharm"]
    )
    
    print(f"\n편의시설 총점 계산 완료 (가중 평균)")
    print(f"  가중치: 편의시설 {W_CONV}, 병원 {W_MED}, 약국 {W_PHARM}")
    print(f"  S_amenity 범위: {listings_valid['S_amenity'].min():.3f} ~ {listings_valid['S_amenity'].max():.3f}")
    print(f"  S_amenity 평균: {listings_valid['S_amenity'].mean():.3f}")
    
    # ========================================
    # 6-3. 편의시설온도 변환 (30~43°C)
    # ========================================
    
    # 공식: AmenityTemp = 30 + 13 × S_amenity
    # 
    # 온도 의미:
    # - S_amenity = 0   → 30.0°C (최저, 편의시설 접근 매우 부족)
    # - S_amenity = 0.5 → 36.5°C (평균 체온, 보통 수준)
    # - S_amenity = 1.0 → 43.0°C (최고, 편의시설 환경 우수)
    #
    # 왜 30~43°C?
    # - 직관적 이해: 사람 체온(36.5°C)을 기준으로 냉온 표현
    # - 교통/공원온도와 동일한 스케일로 비교 가능
    listings_valid["AmenityTemp"] = 30 + 13 * listings_valid["S_amenity"]
    
    print("\n편의시설온도 계산 완료!")
    print(f"  온도 범위: {listings_valid['AmenityTemp'].min():.1f}°C ~ {listings_valid['AmenityTemp'].max():.1f}°C")
    print(f"  평균 온도: {listings_valid['AmenityTemp'].mean():.1f}°C")
    
    # ========================================
    # 통계 및 분포 확인
    # ========================================
    
    print("\n편의시설온도 통계:")
    print(listings_valid["AmenityTemp"].describe())
    
    # 상위 10개 샘플 출력
    print("\n샘플 결과 (상위 10개):")
    result_cols = ["listing_id", "address", "Need_conv", "S_conv", "S_med", "S_pharm", "AmenityTemp"]
    print(listings_valid[result_cols].head(10))
    
    # ========================================
    # 결과 파일 저장
    # ========================================
    
    output_file = BASE_DIR / "listings_with_amenity_temp.csv"
    listings_valid.to_csv(output_file, index=False, encoding="utf-8-sig")
    print(f"\n✅ 결과를 '{output_file.name}'에 저장했습니다.")
    
    # ========================================
    # 편의시설온도 구간별 분포
    # ========================================
    
    print("\n편의시설온도 구간별 매물 분포:")
    
    # 3개 구간으로 분류
    bins = [30, 34, 39, 43]  # 경계값
    labels = ["낮음 (30-34°C)", "보통 (34-39°C)", "높음 (39-43°C)"]
    
    # 구간 분류
    listings_valid["AmenityTemp_category"] = pd.cut(
        listings_valid["AmenityTemp"], 
        bins=bins, 
        labels=labels,
        include_lowest=True  # 30°C를 포함
    )
    
    # 구간별 매물 수 출력
    distribution = listings_valid["AmenityTemp_category"].value_counts().sort_index()
    print(distribution)
    
    # 해석
    print("\n구간별 해석:")
    print("  [낮음 (30-34°C)]")
    print("    - 매물 옵션 부족 + 주변 편의시설 적음")
    print("    - 세탁소, 식당 등 외부 시설 필요하지만 접근 어려움")
    print("    - 생활 편의성 낮음")
    print("\n  [보통 (34-39°C)]")
    print("    - 일반적인 주거 환경")
    print("    - 매물 옵션 또는 주변 편의시설 중 하나는 양호")
    print("    - 기본적인 생활 가능")
    print("\n  [높음 (39-43°C)]")
    print("    - 풀옵션 매물 + 주변 편의시설 풍부")
    print("    - 편의점, 식당, 병원, 약국 모두 가까움")
    print("    - 생활 편의성 매우 우수")
    
else:
    print("⚠️ 유효한 매물이 없습니다.")


6. 최종 편의시설온도 계산
유형별 점수 계산 완료 (S = Need × Access)
  S_conv (편의시설): 평균 0.713
  S_med (병원): 평균 0.626
  S_pharm (약국): 평균 0.612

편의시설 총점 계산 완료 (가중 평균)
  가중치: 편의시설 0.5, 병원 0.25, 약국 0.25
  S_amenity 범위: 0.618 ~ 0.756
  S_amenity 평균: 0.666

편의시설온도 계산 완료!
  온도 범위: 38.0°C ~ 39.8°C
  평균 온도: 38.7°C

편의시설온도 통계:
count     6.000000
mean     38.657895
std       0.615461
min      38.030677
25%      38.441321
50%      38.450955
75%      38.678006
max      39.822981
Name: AmenityTemp, dtype: float64

샘플 결과 (상위 10개):
  listing_id                        address  Need_conv    S_conv     S_med  \
0   18446002      서울특별시 서초구 양재동 116-3 소호M타워        0.7  0.685302  0.583593   
1   18350009     서울특별시 서초구 양재동 208-9 화평빌라타운        0.9  0.872873  0.638355   
2   18448077  서울특별시 서초구 양재동 257-8 다비치 스위트 홈        0.7  0.684186  0.643884   
3   18320532       서울특별시 서초구 양재동 342-8 리더스빌        0.7  0.679790  0.646252   
4   18306808            서울특별시 서초구 양재동 366-9        0.7  0.672775  0.597688   
5   18443369  서울특별시 서초구 양재동 

In [ ]:
# """
# ===================================================================================
# 9. 추가 분석 - 자치구별 편의시설온도 통계
# ===================================================================================
# 목적:
# 1. 자치구별 편의시설온도 평균/분포 비교
# 2. 지역별 편의시설 환경 특성 파악
# 3. 편의시설온도 상위/하위 매물 발굴

# 분석 내용:
# - 자치구별 집계: 매물수, 평균온도, 표준편차, 최소/최대온도
# - 유형별 점수: 편의시설/병원/약국 점수 평균
# - 상위/하위 매물: 편의시설온도 기준 Top 10 / Bottom 10
# ===================================================================================
# """

# print("\n" + "=" * 60)
# print("7. 자치구별 편의시설온도 통계")
# print("=" * 60)

# if len(listings_valid) > 0:
#     # ========================================
#     # 7-1. 주소에서 자치구 추출
#     # ========================================
    
#     def extract_district(address):
#         """
#         주소 문자열에서 자치구 추출
        
#         Args:
#             address: 전체 주소 (예: "서울특별시 서초구 양재동 116-3 소호M타워")
        
#         Returns:
#             자치구명 (예: "서초구")
        
#         로직:
#         - 공백으로 주소 분리
#         - "구"로 끝나는 단어 찾기
#         - 없으면 "기타" 반환
#         """
#         parts = address.split()
#         for part in parts:
#             if part.endswith("구"):
#                 return part
#         return "기타"
    
#     # 매물별 자치구 추출
#     listings_valid["자치구"] = listings_valid["address"].apply(extract_district)
    
#     print("자치구 추출 완료")
#     print(f"자치구 분포:\n{listings_valid['자치구'].value_counts()}")
    
#     # ========================================
#     # 7-2. 자치구별 통계 계산
#     # ========================================
    
#     # groupby + agg로 자치구별 집계
#     district_stats = listings_valid.groupby("자치구").agg({
#         # 편의시설온도 통계
#         "AmenityTemp": [
#             "count",  # 매물 수
#             "mean",   # 평균 온도
#             "std",    # 표준 편차 (온도 분포)
#             "min",    # 최소 온도
#             "max"     # 최대 온도
#         ],
#         # 유형별 점수 평균
#         "S_conv": "mean",    # 편의시설 점수
#         "S_med": "mean",     # 병원 점수
#         "S_pharm": "mean"    # 약국 점수
#     }).round(2)  # 소수점 2자리까지
    
#     # 컬럼명 변경 (읽기 쉽게)
#     district_stats.columns = [
#         "매물수", "평균온도", "표준편차", "최소온도", "최대온도", 
#         "편의시설점수", "병원점수", "약국점수"
#     ]
    
#     # 평균온도 기준 내림차순 정렬
#     district_stats = district_stats.sort_values("평균온도", ascending=False)
    
#     print("\n자치구별 편의시설온도 통계:")
#     print(district_stats)
    
#     # 통계 해석
#     print("\n해석:")
#     top_district = district_stats.index[0]
#     top_temp = district_stats.loc[top_district, "평균온도"]
#     print(f"  편의시설온도 1위: {top_district} ({top_temp:.1f}°C)")
#     print(f"    → 편의시설, 병원, 약국 환경이 가장 우수한 지역")
    
#     if len(district_stats) > 1:
#         bottom_district = district_stats.index[-1]
#         bottom_temp = district_stats.loc[bottom_district, "평균온도"]
#         print(f"\n  편의시설온도 최하위: {bottom_district} ({bottom_temp:.1f}°C)")
#         print(f"    → 편의시설 접근성 개선이 필요한 지역")
    
#     # ========================================
#     # 7-3. 결과 저장
#     # ========================================
    
#     district_stats.to_csv(BASE_DIR / "amenity_temp_by_district.csv", encoding="utf-8-sig")
#     print("\n✅ 자치구별 통계를 'amenity_temp_by_district.csv'에 저장했습니다.")
    
#     # ========================================
#     # 7-4. 편의시설온도 상위/하위 매물
#     # ========================================
    
#     print("\n" + "=" * 60)
#     print("편의시설온도 상위/하위 매물")
#     print("=" * 60)
    
#     # 상위 10개 매물 (편의시설온도 높음)
#     print("\n[편의시설온도 상위 10개 매물]")
#     print("(생활 편의성이 가장 우수한 매물)")
    
#     top_listings = listings_valid.nlargest(10, "AmenityTemp")
#     display_cols = ["listing_id", "address", "자치구", "Need_conv", "AmenityTemp"]
#     print(top_listings[display_cols].to_string(index=False))
    
#     # 상위 매물 특성 분석
#     print("\n상위 매물 특성:")
#     print(f"  평균 온도: {top_listings['AmenityTemp'].mean():.1f}°C")
#     print(f"  평균 편의시설 점수: {top_listings['S_conv'].mean():.2f}")
#     print(f"  평균 병원 점수: {top_listings['S_med'].mean():.2f}")
#     print(f"  평균 약국 점수: {top_listings['S_pharm'].mean():.2f}")
    
#     # 하위 10개 매물 (편의시설온도 낮음)
#     print("\n[편의시설온도 하위 10개 매물]")
#     print("(생활 편의성 개선이 필요한 매물)")
    
#     bottom_listings = listings_valid.nsmallest(10, "AmenityTemp")
#     print(bottom_listings[display_cols].to_string(index=False))
    
#     # 하위 매물 특성 분석
#     print("\n하위 매물 특성:")
#     print(f"  평균 온도: {bottom_listings['AmenityTemp'].mean():.1f}°C")
#     print(f"  평균 편의시설 점수: {bottom_listings['S_conv'].mean():.2f}")
#     print(f"  평균 병원 점수: {bottom_listings['S_med'].mean():.2f}")
#     print(f"  평균 약국 점수: {bottom_listings['S_pharm'].mean():.2f}")
    
#     # ========================================
#     # 7-5. 추가 인사이트
#     # ========================================
    
#     print("\n" + "=" * 60)
#     print("추가 인사이트")
#     print("=" * 60)
    
#     # 필요도와 온도의 관계
#     print("\n1. 매물 옵션과 편의시설온도의 관계:")
#     need_high = listings_valid[listings_valid["Need_conv"] >= 0.9]
#     need_low = listings_valid[listings_valid["Need_conv"] <= 0.7]
    
#     print(f"  옵션 부족 매물 (Need≥0.9): {len(need_high)}개")
#     print(f"    평균 온도: {need_high['AmenityTemp'].mean():.1f}°C")
#     print(f"    → 옵션이 부족하지만 접근성이 좋으면 온도 높음")
    
#     print(f"\n  풀옵션 매물 (Need≤0.7): {len(need_low)}개")
#     print(f"    평균 온도: {need_low['AmenityTemp'].mean():.1f}°C")
#     print(f"    → 옵션이 좋아도 접근성이 낮으면 온도 낮음")
    
#     # 유형별 점수 기여도
#     print("\n2. 유형별 점수 기여도:")
#     print(f"  편의시설 평균 기여: {(listings_valid['S_conv'] * 0.5).mean():.3f}")
#     print(f"  병원 평균 기여: {(listings_valid['S_med'] * 0.25).mean():.3f}")
#     print(f"  약국 평균 기여: {(listings_valid['S_pharm'] * 0.25).mean():.3f}")
#     print(f"  → 편의시설이 전체 점수의 절반을 차지")
    
#     # 온도 범위 분석
#     print("\n3. 온도 범위 분석:")
#     temp_range = listings_valid['AmenityTemp'].max() - listings_valid['AmenityTemp'].min()
#     print(f"  온도 범위: {temp_range:.1f}°C")
#     print(f"  표준편차: {listings_valid['AmenityTemp'].std():.2f}°C")
    
#     if temp_range < 3:
#         print(f"  → 매물 간 편의시설 환경 차이가 작음 (비슷한 지역)")
#     elif temp_range < 7:
#         print(f"  → 매물 간 편의시설 환경 차이가 보통")
#     else:
#         print(f"  → 매물 간 편의시설 환경 차이가 큼 (다양한 지역)")
    
#     print("\n" + "=" * 60)
#     print("✅ 편의시설온도 계산 및 분석 완료!")
#     print("=" * 60)
    
# else:
#     print("⚠️ 유효한 매물이 없습니다.")